# Fine-tune Whisper on MUCS 2021 Hinglish (overnight, Save & Run All)

**Settings:** Accelerator **GPU T4 ×2**, Internet **On**, Input **mucs-prep** (Add Input → Your Work). Then **Save Version → Save & Run All**.

Runs:
- GPU 0: whisper-base on 8 h, then whisper-base on 30 h
- GPU 1: whisper-small on 30 h

All three use the same 2000 steps × batch 32, and the best checkpoint is picked by dev WER. Each model is then scored on the same `eval_300` with greedy decoding and with `no_repeat_ngram_size=5`.

In [ ]:
!nvidia-smi -L
!pip install -q jiwer evaluate indic-transliteration
!df -h /kaggle/working | tail -1

In [ ]:
%%writefile eval.py
"""Evaluate a Whisper checkpoint on a fixed jsonl split (MUCS 2021 Hinglish).

Corpus-level metrics:
  wer / cer                    Devanagari-safe normalization (NFC, lowercase, strip P*/S*)
  wer_romanized / cer_rom.     Devanagari words transliterated to ITRANS first
  pred_script_share            majority script of each prediction (devanagari/latin/arabic/other/empty)
  loop_share_pred / _ref       share of texts containing a repetition loop (heuristic; _ref = false-positive check)
  rtf                          wall time / audio time on --device, batched, after warm-up
"""
import argparse, json, os, re, time, unicodedata
from collections import Counter

import evaluate, jiwer, soundfile as sf, torch
from transformers import pipeline
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate


def norm(t):
    t = unicodedata.normalize("NFC", t).lower().replace("।", " ")
    # strip punctuation/symbols but KEEP Devanagari vowel signs (category M);
    # Whisper's BasicTextNormalizer deletes them
    t = "".join(" " if unicodedata.category(c)[0] in "PS" else c for c in t)
    return " ".join(t.split())


def is_deva(w):
    return any("\u0900" <= c <= "\u097f" for c in w)


# ITRANS has no mapping for candra vowels (ॉ ऑ ॅ ऍ), which Hindi uses mainly for
# English loanwords ("ऑफिस"); left alone they'd be stripped below ("phisa").
DEVA_FIX = str.maketrans({"ऑ": "ओ", "ॉ": "ो", "ऍ": "ए", "ॅ": "े"})


def romanize(t):
    words = []
    for w in norm(t).split():
        if is_deva(w):
            w = transliterate(w.translate(DEVA_FIX), sanscript.DEVANAGARI, sanscript.ITRANS)
            # word-final schwa deletion: ITRANS writes the inherent vowel as lowercase
            # "a" (long आ is "A"), so "करना"->"karanA" keeps it, "ऑफिस"->"ophisa"->"ophis"
            if len(w) > 2 and w[-1] == "a" and w[-2].lower() not in "aeiou":
                w = w[:-1]
            w = w.lower()
        words.append(re.sub(r"[^a-z0-9]", "", w))
    return " ".join(w for w in words if w)


ARABIC = [(0x0600, 0x06FF), (0x0750, 0x077F), (0xFB50, 0xFDFF), (0xFE70, 0xFEFF)]


def script_of(t):
    c = Counter()
    for ch in t:
        o = ord(ch)
        if 0x0900 <= o <= 0x097F:
            c["devanagari"] += 1
        elif any(a <= o <= b for a, b in ARABIC):
            c["arabic"] += 1
        elif ch.isascii() and ch.isalpha():
            c["latin"] += 1
        elif ch.isalpha():
            c["other"] += 1
    return c.most_common(1)[0][0] if c else "empty"


# a 2-30 char unit repeated 4+ times in a row ("लिविविविवि", "click click click click click")
LOOP = re.compile(r"(.{2,30}?)\1{3,}")


def has_loop(t):
    return bool(LOOP.search(t))


def share(labels):
    c = Counter(labels)
    return {k: round(v / len(labels), 4) for k, v in c.most_common()}


def utt_wer(r, p):
    return 1.0 if not p else jiwer.wer(r, p)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="openai/whisper-base")
    ap.add_argument("--data", default="data/eval_300.jsonl")
    ap.add_argument("--n", type=int, default=300)
    ap.add_argument("--device", default="cuda:0" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--threads", type=int, default=4, help="CPU only")
    ap.add_argument("--batch_size", type=int, default=8)
    ap.add_argument("--max_new_tokens", type=int, default=440,
                    help="decoder has 448 positions incl. 4 prompt tokens; Devanagari costs "
                         "several byte-BPE tokens per word, so a low cap truncates real output")
    ap.add_argument("--no_repeat_ngram", type=int, default=0, help="0 = off (plain greedy)")
    ap.add_argument("--tag", default=None, help="output prefix; defaults to the model name")
    args = ap.parse_args()

    tag = args.tag or args.model.strip("/").replace("/", "_")
    if args.device == "cpu":
        torch.set_num_threads(args.threads)
    dtype = torch.float16 if args.device.startswith("cuda") else torch.float32

    rows_all = [json.loads(l) for l in open(args.data, encoding="utf-8")][: args.n]
    # jiwer rejects empty references; drop them identically for every model
    rows = [r for r in rows_all if norm(r["text"]) and romanize(r["text"])]
    n_dropped = len(rows_all) - len(rows)

    asr = pipeline("automatic-speech-recognition", model=args.model,
                   device=args.device, torch_dtype=dtype)
    gen = {"language": "hindi", "task": "transcribe", "max_new_tokens": args.max_new_tokens}
    if args.no_repeat_ngram:
        gen["no_repeat_ngram_size"] = args.no_repeat_ngram

    def load(path):
        a, sr = sf.read(path, dtype="float32")
        assert sr == 16000 and a.ndim == 1, path
        return {"raw": a, "sampling_rate": sr}

    # warm-up on fresh dicts (the pipeline may pop keys from its inputs)
    asr([load(rows[0]["path"]), load(rows[1]["path"])], batch_size=2, generate_kwargs=gen)

    audio = [load(r["path"]) for r in rows]
    t0 = time.perf_counter()
    out = asr(audio, batch_size=args.batch_size, generate_kwargs=gen)
    elapsed = time.perf_counter() - t0

    preds = [o["text"] for o in out]
    refs = [r["text"] for r in rows]
    P, R = [norm(p) for p in preds], [norm(r) for r in refs]
    Pr, Rr = [romanize(p) for p in preds], [romanize(r) for r in refs]
    audio_sec = sum(r["duration"] for r in rows)

    wer, cer = evaluate.load("wer"), evaluate.load("cer")
    metrics = {
        "tag": tag, "model": args.model, "device": args.device, "dtype": str(dtype).split(".")[-1],
        "n": len(rows), "n_dropped_empty_ref": n_dropped,
        "audio_min": round(audio_sec / 60, 2),
        "decoding": {k: v for k, v in gen.items()} | {"batch_size": args.batch_size},
        "wer": round(wer.compute(predictions=P, references=R), 4),
        "cer": round(cer.compute(predictions=P, references=R), 4),
        "wer_romanized": round(wer.compute(predictions=Pr, references=Rr), 4),
        "cer_romanized": round(cer.compute(predictions=Pr, references=Rr), 4),
        "pred_script_share": share([script_of(p) for p in preds]),
        "ref_script_share": share([script_of(r) for r in refs]),
        "loop_share_pred": round(sum(map(has_loop, preds)) / len(preds), 4),
        "loop_share_ref": round(sum(map(has_loop, refs)) / len(refs), 4),
        "rtf": round(elapsed / audio_sec, 4),
    }

    os.makedirs("results", exist_ok=True)
    with open(f"results/{tag}_preds.jsonl", "w", encoding="utf-8") as f:
        for r, p, pn, rn, pr, rr in zip(rows, preds, P, R, Pr, Rr):
            f.write(json.dumps({
                "id": r["id"], "speaker": r["speaker"], "duration": r["duration"],
                "ref": r["text"], "pred": p,
                "wer": round(utt_wer(rn, pn), 4), "wer_romanized": round(utt_wer(rr, pr), 4),
                "pred_script": script_of(p), "loop": has_loop(p),
            }, ensure_ascii=False) + "\n")
    with open(f"results/{tag}_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print(json.dumps(metrics, indent=2, ensure_ascii=False))
    for r, p in list(zip(refs, preds))[:5]:
        print("\nREF :", r, "\nPRED:", p)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train.py
"""Fine-tune Whisper on MUCS 2021 Hinglish jsonl splits (plain PyTorch loop, fp16 AMP).

Writes <out>/best (model + processor, chosen by dev WER), <out>/log.jsonl, <out>/summary.json.
Labels are the references as-is (mixed Devanagari/Latin), so the model learns the dataset's
own script convention.
"""
import argparse, json, math, os, random, time

import numpy as np, soundfile as sf, torch
from torch.utils.data import DataLoader, Dataset
from transformers import WhisperForConditionalGeneration, WhisperProcessor

import jiwer
import unicodedata


def norm(t):  # identical to eval.py's normalizer
    t = unicodedata.normalize("NFC", t).lower().replace("।", " ")
    t = "".join(" " if unicodedata.category(c)[0] in "PS" else c for c in t)
    return " ".join(t.split())


class Clips(Dataset):
    def __init__(self, rows, root, fe, tok, max_label_len=448):
        self.root, self.fe, self.tok = root, fe, tok
        self.rows, self.labels = [], []
        dropped = 0
        for r in rows:
            ids = tok(r["text"]).input_ids
            if len(ids) > max_label_len:
                dropped += 1; continue
            self.rows.append(r); self.labels.append(ids)
        if dropped:
            print(f"dropped {dropped} rows with > {max_label_len} label tokens", flush=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        a, sr = sf.read(os.path.join(self.root, r["path"]), dtype="float32")
        assert sr == 16000, r["path"]
        feats = self.fe(a, sampling_rate=16000, return_tensors="np").input_features[0]
        return feats.astype(np.float32), self.labels[i], i


class Collate:
    def __init__(self, start_id):
        self.start_id = start_id

    def __call__(self, batch):
        feats = torch.from_numpy(np.stack([b[0] for b in batch]))
        labs = [b[1][1:] if b[1][0] == self.start_id else b[1] for b in batch]  # model re-adds SOT
        L = max(map(len, labs))
        labels = torch.full((len(labs), L), -100, dtype=torch.long)
        for j, l in enumerate(labs):
            labels[j, :len(l)] = torch.tensor(l)
        return feats, labels, [b[2] for b in batch]


def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8")]


@torch.no_grad()
def evaluate_dev(model, loader, tok, dev, device, max_new_tokens):
    model.eval()
    preds, refs = [], []
    for feats, _, idx in loader:
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            out = model.generate(input_features=feats.to(device), language="hindi", task="transcribe",
                                 max_new_tokens=max_new_tokens)
        preds += tok.batch_decode(out, skip_special_tokens=True)
        refs += [dev.rows[i]["text"] for i in idx]
    model.train()
    P, R = [norm(p) for p in preds], [norm(r) for r in refs]
    return jiwer.wer(R, P), list(zip(refs[:3], preds[:3]))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="openai/whisper-base")
    ap.add_argument("--root", required=True, help="dir that jsonl 'path' fields are relative to")
    ap.add_argument("--train", required=True)
    ap.add_argument("--dev", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--max_steps", type=int, default=2000)
    ap.add_argument("--batch_size", type=int, default=32, help="effective batch")
    ap.add_argument("--micro_batch", type=int, default=32)
    ap.add_argument("--lr", type=float, default=2.5e-5)
    ap.add_argument("--warmup", type=int, default=200)
    ap.add_argument("--eval_every", type=int, default=500)
    ap.add_argument("--dev_n", type=int, default=0, help="0 = whole dev set")
    ap.add_argument("--eval_max_new_tokens", type=int, default=225)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--spec_augment", type=int, default=1)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()
    assert args.batch_size % args.micro_batch == 0
    accum = args.batch_size // args.micro_batch

    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(args.out, exist_ok=True)

    proc = WhisperProcessor.from_pretrained(args.model, language="hindi", task="transcribe")
    tok, fe = proc.tokenizer, proc.feature_extractor
    model = WhisperForConditionalGeneration.from_pretrained(args.model)
    model.config.forced_decoder_ids = None
    model.generation_config.forced_decoder_ids = None
    model.generation_config.language = "hindi"
    model.generation_config.task = "transcribe"
    if args.spec_augment:
        model.config.apply_spec_augment = True
        model.config.mask_time_prob = 0.05
    model.to(device).train()

    train_rows, dev_rows = read_jsonl(args.train), read_jsonl(args.dev)
    if args.dev_n:
        dev_rows = dev_rows[:args.dev_n]
    train = Clips(train_rows, args.root, fe, tok)
    dev = Clips(dev_rows, args.root, fe, tok)
    collate = Collate(model.config.decoder_start_token_id)
    kw = dict(num_workers=args.workers, collate_fn=collate, pin_memory=device.type == "cuda",
              persistent_workers=args.workers > 0)
    train_dl = DataLoader(train, batch_size=args.micro_batch, shuffle=True, drop_last=True, **kw)
    dev_dl = DataLoader(dev, batch_size=args.micro_batch, shuffle=False, **kw)

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.0)
    def lr_at(s):  # linear warmup, then linear decay to 0
        if s < args.warmup:
            return (s + 1) / args.warmup
        return max(0.0, (args.max_steps - s) / max(1, args.max_steps - args.warmup))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    hours = sum(r["duration"] for r in train.rows) / 3600
    epochs = args.max_steps * args.batch_size / len(train)
    print(f"train {len(train)} utts ({hours:.2f} h), dev {len(dev)}; {args.max_steps} steps x {args.batch_size} "
          f"= {epochs:.1f} epochs; accum {accum}; device {device}", flush=True)

    log = open(f"{args.out}/log.jsonl", "a")
    best, step, t0, run_loss = math.inf, 0, time.time(), 0.0

    def do_eval():
        nonlocal best
        te = time.time()
        w, ex = evaluate_dev(model, dev_dl, tok, dev, device, args.eval_max_new_tokens)
        rec = {"step": step, "dev_wer": round(w, 4), "eval_sec": round(time.time() - te, 1),
               "elapsed_min": round((time.time() - t0) / 60, 1)}
        if w < best:
            best = w
            model.save_pretrained(f"{args.out}/best")
            proc.save_pretrained(f"{args.out}/best")
            rec["saved"] = True
        print(json.dumps(rec), flush=True)
        for r, p in ex:
            print("  REF :", r, "\n  PRED:", p, flush=True)
        log.write(json.dumps(rec) + "\n"); log.flush()

    do_eval()  # step 0 = the pretrained model on dev, for the training curve
    it = iter(train_dl)
    while step < args.max_steps:
        for _ in range(accum):
            try:
                feats, labels, _ = next(it)
            except StopIteration:
                it = iter(train_dl)
                feats, labels, _ = next(it)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                loss = model(input_features=feats.to(device, non_blocking=True),
                             labels=labels.to(device, non_blocking=True)).loss / accum
            scaler.scale(loss).backward()
            run_loss += loss.item()
        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        step += 1
        if step % 50 == 0 or step == 1:
            rec = {"step": step, "loss": round(run_loss / (1 if step == 1 else 50), 4),
                   "lr": sched.get_last_lr()[0], "grad_norm": round(float(gn), 3),
                   "sec_per_step": round((time.time() - t0) / step, 3)}
            print(json.dumps(rec), flush=True)
            log.write(json.dumps(rec) + "\n"); log.flush()
            run_loss = 0.0
        if step % args.eval_every == 0 or step == args.max_steps:
            do_eval()

    summary = {"model": args.model, "train": args.train, "train_utts": len(train),
               "train_hours": round(hours, 2), "steps": args.max_steps, "batch_size": args.batch_size,
               "epochs": round(epochs, 2), "lr": args.lr, "best_dev_wer": round(best, 4),
               "train_minutes": round((time.time() - t0) / 60, 1)}
    json.dump(summary, open(f"{args.out}/summary.json", "w"), indent=2)
    print("DONE", json.dumps(summary), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
import glob, os, json, subprocess, hashlib
tj = glob.glob("/kaggle/input/**/mucs_prep/train_30h.jsonl", recursive=True)
ej = glob.glob("/kaggle/input/**/data/eval_300.jsonl", recursive=True)
assert tj and ej, "attach the mucs-prep notebook output as input (Add Input -> Your Work)"
ROOT = os.path.dirname(tj[0]); DATA = os.path.dirname(ej[0])
print("ROOT", ROOT, "\nDATA", DATA)
# eval.py expects data/test_segments/... relative paths; symlink instead of copying
os.makedirs("data", exist_ok=True)
for n in ["test_segments", "eval_300.jsonl", "test_all.jsonl"]:
    if not os.path.exists(f"data/{n}"):
        os.symlink(f"{DATA}/{n}", f"data/{n}")
h = hashlib.md5("".join(json.loads(l)["id"] for l in open("data/eval_300.jsonl")).encode()).hexdigest()
assert h == "2699562b2ab929891ecdd2f7cf0816e6", h
print("eval split OK;", json.load(open(f"{ROOT}/stats.json"))["train_30h.jsonl"])
import torch; NGPU = torch.cuda.device_count(); print("GPUs:", NGPU)

## Smoke test
Twelve steps on 16 dev clips. It catches setup errors in about 3 minutes instead of hours into the run.

In [ ]:
r = subprocess.run(f"python train.py --model openai/whisper-base --root {ROOT} --train {ROOT}/train_8h.jsonl "
                   f"--dev {ROOT}/dev.jsonl --out /tmp/smoke --max_steps 12 --warmup 4 --eval_every 6 --dev_n 16 "
                   f"--batch_size 16 --micro_batch 8", shell=True, capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])
assert r.returncode == 0 and os.path.exists("/tmp/smoke/summary.json"), "smoke test failed: see above"
print("SMOKE TEST PASSED")

## Training

In [ ]:
os.makedirs("logs", exist_ok=True)
def cmd(name, model, train, lr, micro):
    return (f"python train.py --model {model} --root {ROOT} --train {ROOT}/{train} --dev {ROOT}/dev.jsonl "
            f"--out runs/{name} --max_steps 2000 --batch_size 32 --micro_batch {micro} --lr {lr} "
            f"--eval_every 500 > logs/{name}.log 2>&1")
base8  = cmd("base_8h",  "openai/whisper-base",  "train_8h.jsonl",  2.5e-5, 16)
base30 = cmd("base_30h", "openai/whisper-base",  "train_30h.jsonl", 2.5e-5, 16)
small30 = cmd("small_30h", "openai/whisper-small", "train_30h.jsonl", 1e-5, 8)
if NGPU >= 2:
    script = (f"(CUDA_VISIBLE_DEVICES=0 {base8}; CUDA_VISIBLE_DEVICES=0 {base30}) & "
              f"(CUDA_VISIBLE_DEVICES=1 {small30}) & wait")
else:
    script = f"{base8}; {base30}; {small30}"
subprocess.run(script, shell=True, executable="/bin/bash")
for n in ["base_8h", "base_30h", "small_30h"]:
    print(f"===== {n}"); print(subprocess.run(f"tail -n 25 logs/{n}.log", shell=True, capture_output=True, text=True).stdout)

## Evaluate on eval_300 (same 300 clips as the baselines)

In [ ]:
for n in ["base_8h", "base_30h", "small_30h"]:
    if not os.path.exists(f"runs/{n}/best"):
        print("SKIP (no checkpoint):", n); continue
    for extra, tag in [("", f"ft_{n}"), ("--no_repeat_ngram 5", f"ft_{n}_nrng5")]:
        subprocess.run(f"python eval.py --model runs/{n}/best --device cuda:0 --tag {tag} {extra} "
                       f"> logs/eval_{tag}.log 2>&1", shell=True)
        print(tag, subprocess.run(f"tail -n 3 logs/eval_{tag}.log", shell=True, capture_output=True, text=True).stdout[-300:])

In [ ]:
import pandas as pd
rows = []
for f in sorted(glob.glob("results/*_metrics.json")):
    m = json.load(open(f))
    rows.append({k: m[k] for k in ["tag", "wer", "cer", "wer_romanized", "cer_romanized", "loop_share_pred", "rtf"]}
                | {"pred_" + k: v for k, v in m["pred_script_share"].items()})
df = pd.DataFrame(rows).set_index("tag").fillna(0); print(df.to_string()); df

In [ ]:
curves = {n: [json.loads(l) for l in open(f"runs/{n}/log.jsonl")] for n in ["base_8h", "base_30h", "small_30h"]
          if os.path.exists(f"runs/{n}/log.jsonl")}
for n, c in curves.items():
    print(n, [(r["step"], r["dev_wer"]) for r in c if "dev_wer" in r])
!zip -qr results_train.zip results logs runs/*/log.jsonl runs/*/summary.json && ls -lh results_train.zip
!du -sh runs/*/best